In [1]:
import os


In [2]:
import cv2

In [3]:
from keras.preprocessing import image

In [4]:
categories = ['with_mask','without_mask']

In [5]:
import numpy as np

In [6]:
data = []

for category in categories:
    path = os.path.join('train',category)

    label = categories.index(category)
    
    for file in os.listdir(path):

        img_path= os.path.join(path,file)
        img = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8),cv2.IMREAD_COLOR)

        if img is None:
            print("Cannot read:", img_path)
            continue
        
        img = cv2.resize(img,(224,224))
        
        
        data.append([img,label])
        

In [7]:
import random

In [8]:
random.shuffle(data)

In [9]:
x=[]
y=[]

for features, label in data:
    x.append(features)
    y.append(label)

In [10]:
len(x)

4095

In [11]:
import numpy as np

In [12]:
x = np.array(x)
y = np.array(y)

In [13]:
x.shape

(4095, 224, 224, 3)

In [14]:
y.shape

(4095,)

In [15]:
y

array([1, 0, 0, ..., 0, 1, 1], shape=(4095,))

In [16]:
X =x/255

In [17]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

In [18]:
X_train.shape

(3276, 224, 224, 3)

In [19]:
X_test.shape

(819, 224, 224, 3)

In [20]:
from keras.applications.vgg16 import VGG16

In [21]:
vgg = VGG16()

In [22]:
vgg.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv1 (Conv2D)         │ (None, 224, 224, 64)  │        1,792 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv2 (Conv2D)         │ (None, 224, 224, 64)  │       36,928 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_pool (MaxPooling2D)    │ (None, 112, 112, 64)  │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv1 (Conv2D)         │ (None, 112, 112, 128) │       73,856 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv2 (Conv2D)         │ (None, 112, 112, 128) │      147,584 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 56, 56, 128)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv1 (Conv2D)         │ (None, 56, 56, 256)   │      295,168 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv2 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv3 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_pool (MaxPooling2D)    │ (None, 28, 28, 256)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv1 (Conv2D)         │ (None, 28, 28, 512)   │    1,180,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv2 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv3 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_pool (MaxPooling2D)    │ (None, 14, 14, 512)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv1 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv2 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv3 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_pool (MaxPooling2D)    │ (None, 7, 7, 512)     │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ flatten (Flatten)             │ (None, 25088)         │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc1 (Dense)                   │ (None, 4096)          │  102,764,544 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc2 (Dense)                   │ (None, 4096)          │   16,781,312 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ predictions (Dense)           │ (None, 1000)          │    4,097,000 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 138,357,544 (527.79 MB)

 Trainable params: 138,357,544 (527.79 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
from keras  import Sequential

In [24]:
model = Sequential()

In [25]:
for layer in vgg.layers[:-1]:
    model.add(layer)

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)         │ (None, 224, 224, 64)  │        1,792 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv2 (Conv2D)         │ (None, 224, 224, 64)  │       36,928 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_pool (MaxPooling2D)    │ (None, 112, 112, 64)  │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv1 (Conv2D)         │ (None, 112, 112, 128) │       73,856 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv2 (Conv2D)         │ (None, 112, 112, 128) │      147,584 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 56, 56, 128)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv1 (Conv2D)         │ (None, 56, 56, 256)   │      295,168 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv2 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv3 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_pool (MaxPooling2D)    │ (None, 28, 28, 256)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv1 (Conv2D)         │ (None, 28, 28, 512)   │    1,180,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv2 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv3 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_pool (MaxPooling2D)    │ (None, 14, 14, 512)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv1 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv2 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv3 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_pool (MaxPooling2D)    │ (None, 7, 7, 512)     │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ flatten (Flatten)             │ (None, 25088)         │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc1 (Dense)                   │ (None, 4096)          │  102,764,544 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc2 (Dense)                   │ (None, 4096)          │   16,781,312 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 134,260,544 (512.16 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
for layer in model.layers:
    layer.trainable=False
    

In [28]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)         │ (None, 224, 224, 64)  │        1,792 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv2 (Conv2D)         │ (None, 224, 224, 64)  │       36,928 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_pool (MaxPooling2D)    │ (None, 112, 112, 64)  │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv1 (Conv2D)         │ (None, 112, 112, 128) │       73,856 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv2 (Conv2D)         │ (None, 112, 112, 128) │      147,584 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 56, 56, 128)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv1 (Conv2D)         │ (None, 56, 56, 256)   │      295,168 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv2 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv3 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_pool (MaxPooling2D)    │ (None, 28, 28, 256)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv1 (Conv2D)         │ (None, 28, 28, 512)   │    1,180,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv2 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv3 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_pool (MaxPooling2D)    │ (None, 14, 14, 512)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv1 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv2 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv3 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_pool (MaxPooling2D)    │ (None, 7, 7, 512)     │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ flatten (Flatten)             │ (None, 25088)         │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc1 (Dense)                   │ (None, 4096)          │  102,764,544 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc2 (Dense)                   │ (None, 4096)          │   16,781,312 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 134,260,544 (512.16 MB)

In [29]:
from keras.layers import Dense

In [30]:
model.add(Dense(1, activation = 'sigmoid'))

In [31]:
model.compile(optimizer='Adam' , loss = 'binary_crossentropy', metrics=['accuracy'])

In [32]:
model.fit(X_train,y_train, epochs=6, validation_data = (X_test,y_test))

Epoch 1/6
103/103 ━━━━━━━━━━━━━━━━━━━━ 265s 3s/step - accuracy: 0.7817 - loss: 0.4859 - val_accuracy: 0.8596 - val_loss: 0.3671
Epoch 2/6
103/103 ━━━━━━━━━━━━━━━━━━━━ 248s 2s/step - accuracy: 0.8929 - loss: 0.3076 - val_accuracy: 0.8864 - val_loss: 0.3001
Epoch 3/6
103/103 ━━━━━━━━━━━━━━━━━━━━ 240s 2s/step - accuracy: 0.9093 - loss: 0.2576 - val_accuracy: 0.8950 - val_loss: 0.2811
Epoch 4/6
103/103 ━━━━━━━━━━━━━━━━━━━━ 230s 2s/step - accuracy: 0.9216 - loss: 0.2277 - val_accuracy: 0.8718 - val_loss: 0.2967
Epoch 5/6
103/103 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.9270 - loss: 0.2069 - val_accuracy: 0.9267 - val_loss: 0.2176
Epoch 6/6
103/103 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.9313 - loss: 0.1929 - val_accuracy: 0.9304 - val_loss: 0.2056


In [45]:
cap = cv2.VideoCapture(0)

In [46]:
import cv2
def draw_label(img,text,pos,bg_color):
    text_size = cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,1,cv2.FILLED)

    end_x = pos[0] + text_size[0][0] + 2
    end_y = pos[1] + text_size[0][1] - 2

    cv2.rectangle(img,pos,(end_x,end_y),bg_color,cv2.FILLED)
    cv2.putText(img,text,pos,cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,0),1,cv2.LINE_AA)

In [57]:
def detect_face_mask(img):
    img = img.reshape(1, 224, 224, 3)
    y_pred = model.predict(img, verbose=0)
    return int(y_pred[0][0] > 0.5)

In [53]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)         │ (None, 224, 224, 64)  │        1,792 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_conv2 (Conv2D)         │ (None, 224, 224, 64)  │       36,928 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block1_pool (MaxPooling2D)    │ (None, 112, 112, 64)  │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv1 (Conv2D)         │ (None, 112, 112, 128) │       73,856 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_conv2 (Conv2D)         │ (None, 112, 112, 128) │      147,584 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 56, 56, 128)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv1 (Conv2D)         │ (None, 56, 56, 256)   │      295,168 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv2 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_conv3 (Conv2D)         │ (None, 56, 56, 256)   │      590,080 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block3_pool (MaxPooling2D)    │ (None, 28, 28, 256)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv1 (Conv2D)         │ (None, 28, 28, 512)   │    1,180,160 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv2 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_conv3 (Conv2D)         │ (None, 28, 28, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block4_pool (MaxPooling2D)    │ (None, 14, 14, 512)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv1 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv2 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_conv3 (Conv2D)         │ (None, 14, 14, 512)   │    2,359,808 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ block5_pool (MaxPooling2D)    │ (None, 7, 7, 512)     │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ flatten (Flatten)             │ (None, 25088)         │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc1 (Dense)                   │ (None, 4096)          │  102,764,544 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ fc2 (Dense)                   │ (None, 4096)          │   16,781,312 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense (Dense)                 │ (None, 1)             │        4,097 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 134,272,837 (512.21 MB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 134,260,544 (512.16 MB)

 Optimizer params: 8,196 (32.02 KB)

In [54]:
haar= cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

In [55]:
def detect_face(img):
    coods = haar.detectMultiScale(img)
    return coods

In [58]:
while True:
    ret, frame = cap.read()

     # Check if frame is received
    if not ret or frame is None:
        print("Failed to capture frame")
        break
        
    # call the detection method
    img = cv2.resize(frame,(224,224))

    y_pred = detect_face_mask(img)

    coods = detect_face(cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY))

    for x,y,w,h in coods:
        cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),3)

    if y_pred == 0:
        draw_label(frame,"Mask",(30,30),(0,255,0))
    else:
        draw_label(frame,"No Mask",(30,30),(0,0,255))
    
    cv2.imshow("window",frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break


cv2.destroyAllWindows()

KeyboardInterrupt: 